# Velocity and Stress Summary Calculations and Figures

This notebook brings in summaries of ADV velocity data that has already been processed, and summarizes it into longitudinal distributions of depth-averaged velocity and bed shear stress upstream and over the nest in each flume run.

The notebook needs:
- a file called ```summary_conditions_locations.csv``` to provide spatial information for the measurement locations
- a file called ```summary_velocity_mean.csv``` which is the natural output from ```ADV_data_01...```
- a file called ```summary_profile_anal_all.csv``` which is the natural output from ```ADV_data_02...```

User input:
- Data folder
- Profile $x$ location

Version/Change log:
- V03: change run names to match those in the paper (swap run 3 and 4)

## Imports and Functions

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob

%config InlineBackend.figure_format='retina' # high-res plots for a Retina display, uncomment while working

plt.style.use('seaborn-v0_8-colorblind') # Use 'tab10' as an alternative

plt.matplotlib.rcParams.update({'font.size': 8})
plt.rcParams['legend.fontsize'] = 7

fullwidth = 6
fullheight = 3

halfwidth = 3.
halfheight = 2.33

# returns the clear-water density in kg/m^3 as a function of temperature [in deg C] and salinity [ppt]
# empirical fit: EOS-80 / UNESCO seawater equation of state (TODO: confirm/insert citation)
def rho_cw(T,S): 
    rho_fresh=1000*(1-(T +288.9414)/(508929.2*(T+68.12963))*(T-3.9863)**2)
    Acoef = 0.824493 - 0.0040899*T + 0.000076438*T**2 -0.00000082467*T**3 + 0.0000000053675*T**4
    Bcoef = -0.005724 + 0.00010227*T - 0.0000016546*T**2
    return rho_fresh + Acoef*S + Bcoef*S**(3/2) + 0.00048314*S**2 

# returns the clear-water kinematic viscosity in [m^2/s] as a function of temperature [in deg C].
# empirical fit to tabulated water viscosity (TODO: confirm/insert citation)
def visc(T):
    return 2.7488e-07+1.4907e-06*np.exp(-0.034812*T)


## User input

In [2]:
datadir_input = 'data_input/' # path to the data
datadir_output = 'data_output/flume_vel_stress/' # path to the output data
fig_folder = 'figures/flume_vel_stress'

veldatafile = 'summary_velocity_mean.csv'
conditionlocationfile = 'summary_conditions_locations.csv'
profiledata = 'summary_profile_anal_all.csv'

profilex = 5.12 # x location in meters where the velocity profile is located

T = 20 # water temperature in deg C

## Find and load the data

In [3]:
# Make the output folders
if os.path.isdir(fig_folder) != 1:
    os.mkdir(fig_folder)
if os.path.isdir(datadir_output) != 1:
    os.mkdir(datadir_output)

data = pd.read_csv(datadir_input+veldatafile) # reads in the data
info = pd.read_csv(datadir_input+conditionlocationfile) # reads in the data
profiles = pd.read_csv(datadir_input+profiledata)

# add columns to info dataframe 
info.insert(loc=1,column='Run-original',value=info.File.str[0:5])
info.insert(loc=2,column='Point',value=info.File.str[6:8].astype(int))
info.insert(loc=4,column='S',value=info.S_p/100)

runs_og_unique = sorted(info['Run-original'].unique()) # get the unique run 

# Rename the runs to match the paper: the original acquisition order (Run1-Run4) did not
# increase monotonically with bed slope, so remap to a slope-ordered 1-16 sequence.
# Run 3 has the highest slope, so Run3A-D become the last four runs in the numeric order.
run_map = {
    'Run1A': 1,
    'Run1B': 2,
    'Run1C': 3,
    'Run1D': 4,
    'Run2A': 5,
    'Run2B': 6,
    'Run2C': 7,
    'Run2D': 8,
    'Run4A': 9,
    'Run4B': 10,
    'Run4C': 11,
    'Run4D': 12,
    'Run3A': 13,
    'Run3B': 14,
    'Run3C': 15,
    'Run3D': 16
}

# info already has 'Run-original'
info.insert(loc=2, column='Run', value=info['Run-original'].map(run_map))
data.insert(loc=1, column='Run', value=data['Run-original'].map(run_map))
profiles.insert(loc=1, column='Run', value=profiles['Run-original'].map(run_map))

runs = sorted(info['Run'].unique()) # get the unique run 

# display(data)
# display(info)
# display(profiles)


## Process all of the data

looks for profiles, calculates the depth average and then combines it with the rest of the velocity data for that run

In [7]:
summary_rows = []

for j in range(0,len(runs)):
    
    run = runs[j]

    # For each run: depth-average the vertical velocity profile taken at x = profilex,
    # then stitch that single depth-averaged value back into the streamwise (x) series of
    # point measurements so velocity and bed stress can be plotted along the flume.
    rho = rho_cw(T,0)
    output_path=datadir_output+'/summary_vel_x_run'+str(run)+'.csv'

    # pull the profile data ------------------

    profile_summary = profiles[(profiles.Run == run)]
    profile_summary = profile_summary.loc[:, :'uavg_m_s']
    profile_summary['S_p']=profile_summary.S*100
    profile_summary['File']='depth avg'
    profile_summary['Point']='DepthAvg'
    profile_summary = profile_summary.drop(columns=['Run-original'])
    
    Uavg = profile_summary.uavg_m_s.iloc[0]
    # Darcy-Weisbach friction factor f. Three estimates are available; the depth-slope
    # product (f_ds) is used for the paper as the most robust reach-averaged estimate.
    # f=profiles[(profiles.Run == run)].f_uw.iloc[0]        # f_uw = stress via the Reynolds stress
    f=profiles[(profiles.Run == run)].f_ds.iloc[0]          # f_ds = stress via the depth slope product
#     f=profiles[(profiles.Run == run)].f_profile.iloc[0]   # f_profile = stress via the u* fit to the velocity

    # pull the non profile data ------

    npinfo = info[(info.Run == run)&(info.x_m != profilex)].copy() # pull out the non profile for the run
    npinfo = npinfo.sort_values(by=['Point']).reset_index(drop=True) # sort the data on the points in case something is out of order

    npfirst = npinfo.Point.min() # define which points are part of the profile - the first
    nplast = npinfo.Point.max()  # define which points are part of the profile - the last
    npdata = data[(data.Run == run)&(data.Point >= npfirst)&(data.Point <= nplast)].copy() # pull out the profile for the run
    npdata = npdata.sort_values(by=['Point']).reset_index(drop=True)   # sort the data on the points in case something is out of order

    # drop data in the combine... 
    npinfo = npinfo.drop(columns=['Run-original'])
    npdata = npdata.drop(columns=['snr1','snr2','snr3','snr4','cor1','cor2','cor3','cor4'])
    npdata = npdata.drop(columns=['uu_bar','uv_bar','uw_bar','vv_bar','vw_bar','ww_bar'])
    npdata = npdata.drop(columns=['vavg_m_s','wavg_m_s'])
    npdata = npdata.drop(columns=['Run','Point','Run-original'])

    summary = pd.concat([npinfo,npdata], axis=1)
    summary = pd.concat((summary, profile_summary), axis = 0)
    summary = summary.sort_values(by=['x_m']).reset_index(drop=True)

    summary['Fr_local']=summary.uavg_m_s/np.sqrt(9.81*summary.h_cm/100)
    # bed shear stress from the Darcy-Weisbach relation: tau_b = (f/8) * rho * U^2
    summary['tau_b_Pa'] = (f/8)*rho*summary.uavg_m_s**2

    summary.to_csv(output_path, index=False)
    # display(summary)

    UAVGstr = str(np.around(Uavg, decimals=2))
    Umax = str(np.around(summary.uavg_m_s.max(), decimals=2))

    taub_ambient = str(np.around((f/8)*rho*profile_summary.uavg_m_s.iloc[0]**2,decimals=1))
    taub_max = str(np.around(max(summary.tau_b_Pa),decimals=1))
    fric = str(np.around(f,decimals=3))
    
    summary_rows.append({
        'Run': run,
        'Uavg_m_s': Uavg,
        'Umax_m_s': summary.uavg_m_s.max(),
        'f': f,
        'taub_ambient_Pa': (f/8)*rho*profile_summary.uavg_m_s.iloc[0]**2,
        'taub_max_Pa': max(summary.tau_b_Pa),
    })
    
    # Figures Individual ------------

    fig, ax = plt.subplots(figsize=(halfwidth,halfheight))
    ax.plot(-summary.x_m,summary.uavg_m_s, 'o-', alpha=1, label='$U_{avg}$')
    ax.axvline(x=-3.85, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('$x$ [m]')
    ax.set_ylabel('Avg Velocity [m/s]')
    ax.set_ylim(-0.25,1.25)
    ax.text(-5.15,-0.0, '$U_{ambient} = $'+UAVGstr+' m/s', backgroundcolor='none', 
            clip_on='True', multialignment='center', alpha=1) #weight = 'bold', 
    ax.text(-5.15,-0.1, '$U_{peak} = $'+Umax+' m/s', backgroundcolor='none', 
            clip_on='True', multialignment='center', alpha=1) #weight = 'bold', 
    ax.text(-5.15,0.1, 'Run '+str(run), backgroundcolor='none', clip_on='True', multialignment='center', alpha=1) #weight = 'bold', 
    plt.legend(loc=2, ncol=1)
    plt.savefig(fig_folder+'/Run'+str(run)+'_U(x).pdf',bbox_inches="tight", pad_inches=0.005)
    plt.close()

    fig, ax = plt.subplots(figsize=(halfwidth,halfheight))
    ax.plot(-summary.x_m,summary.tau_b_Pa, 'o-', color='C2', alpha=1, label=r'$\tau_{b}$')
    ax.axvline(x=-3.85, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('$x$ [m]')
    ax.set_ylabel('Bed Shear Stress [Pa]')
    ax.set_ylim(0,22)
    ax.set_ylim(bottom=0)
    ax.text(-5.15,20, 'Run '+str(run)+', $f=$'+fric, backgroundcolor='none', clip_on='True', multialignment='center', alpha=1) #weight = 'bold', 
    ax.text(-5.15,18.0, r'$\tau_{b,a} = $'+taub_ambient+' Pa', backgroundcolor='none', 
            clip_on='True', multialignment='center', alpha=1) #weight = 'bold', 
    ax.text(-5.15,16.0, r'$\tau_{b,p} = $'+taub_max+' Pa', backgroundcolor='none', 
            clip_on='True', multialignment='center', alpha=1) #weight = 'bold', 
    plt.legend(loc=1, ncol=1)
    plt.savefig(fig_folder+'/Run'+str(run)+'_tau(x).pdf',bbox_inches="tight", pad_inches=0.005)
    plt.close()
    
run_summary = pd.DataFrame(summary_rows)
run_summary.to_csv(datadir_output+'/summary_u_tau.csv', index=False)
    

In [5]:
import re

def run_sort_key(filename):
    match = re.search(r'run(\d+)([A-Za-z]?)', filename)
    return (int(match.group(1)), match.group(2))

files = os.path.join(datadir_output, "summary_vel_x_run*.csv")
files = glob.glob(files)
files = sorted(files, key=run_sort_key)

df = pd.concat(map(pd.read_csv, files), ignore_index=True)

df.to_csv(datadir_output+'/summary_vel_x_all.csv', index=False)
